# 📄 PDF & Document Reader MVP — Qwen3-TTS
Upload any PDF, Markdown, or .txt file and convert it to narrated audio chapters.

**Supported file types:** `.pdf`, `.md`, `.txt`
**Tips:** Make sure the text is mostly continuous. Too many short list items may sound disjointed.

In [ ]:
!pip install -q qwen-tts soundfile pypdf
import os
import re
import textwrap
import numpy as np
import soundfile as sf
from pypdf import PdfReader
from IPython.display import Audio, display

In [ ]:
MODEL_ID = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
OUTPUT_DIR = "reader_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'(?i)page \d+', '', text)
    return text.strip()

# Option A (default): Hardcoded sample text
USER_TEXT = """Space exploration is the use of astronomy and space technology to explore outer space. 
While the exploration of space is carried out mainly by astronomers with telescopes, its physical exploration though is conducted both by unmanned robotic space probes and human spaceflight.
Space exploration, like its classical form astronomy, is one of the main sources for space science.

The history of space exploration is marked by milestones. 
The first human-made object to reach space was the V-2 rocket in 1944. 
The first artificial satellite was Sputnik 1, launched by the Soviet Union in 1957.
The first human in space was Yuri Gagarin in 1961. 

Future missions will likely return humans to the Moon and establish a long-term presence there. 
Mars is another major target of space exploration, with both crewed and uncrewed missions planned by multiple national and private entities.
The exploration of the solar system and beyond continues to captivate humanity and push the boundaries of technology."""
USER_TEXT = clean_text(USER_TEXT)

# Option B (commented): Upload PDF
# from google.colab import files
# uploaded = files.upload()
# pdf_filename = list(uploaded.keys())[0]
# reader = PdfReader(pdf_filename)
# USER_TEXT = clean_text(" ".join(page.extract_text() for page in reader.pages))

# Option C (commented): Upload .txt or .md file
# from google.colab import files
# uploaded = files.upload()
# txt_filename = list(uploaded.keys())[0]
# with open(txt_filename, "r") as f:
#     USER_TEXT = clean_text(f.read())

print("Text loaded!")

In [ ]:
words = len(USER_TEXT.split())
chars = len(USER_TEXT)
reading_time_mins = words / 150
audio_duration_mins = words / 130
num_chunks = max(1, chars // 280)

print(f"Document Statistics:")
print(f"Words: {words}")
print(f"Characters: {chars}")
print(f"Estimated Reading Time: {reading_time_mins:.2f} minutes")
print(f"Estimated Audio Duration: {audio_duration_mins:.2f} minutes")
print(f"Estimated Chunks: {num_chunks}")

In [ ]:
import torch
from qwen_tts import Qwen3TTSModel

device = "cuda:0" if torch.cuda.is_available() else "cpu"
model = Qwen3TTSModel.from_pretrained(
    MODEL_ID, 
    device_map=device, 
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32, 
    attn_implementation="sdpa"
)
INSTRUCT = "A warm, clear, engaging narrator — measured pace, excellent diction, like a professional audiobook reader"
print("Model loaded successfully!")

In [ ]:
def chunk_text(text, max_chars=280):
    sentences = re.split(r'(?<=[.!?]) +', text)
    chunks = []
    curr = ""
    for s in sentences:
        if len(curr) + len(s) > max_chars and curr:
            chunks.append(curr.strip())
            curr = s + " "
        else:
            curr += s + " "
    if curr:
        chunks.append(curr.strip())
    return chunks

chunks = chunk_text(USER_TEXT)
all_audio = []
sr = 24000 # Default sample rate

for i, chunk in enumerate(chunks):
    print(f"Generating chunk {i+1}/{len(chunks)}...")
    audio, sr = model.generate_voice_design(
        text=chunk,
        language="English",
        instruct=INSTRUCT
    )
    all_audio.append(audio)

final_audio = np.concatenate(all_audio)
out_path = f"{OUTPUT_DIR}/document_narration.wav"
sf.write(out_path, final_audio, sr)
print(f"Saved complete narration to {out_path}")

In [ ]:
duration_sec = len(final_audio) / sr
file_size_mb = os.path.getsize(out_path) / (1024 * 1024)
wpm = words / (duration_sec / 60) if duration_sec > 0 else 0

print(f"Total Audio Duration: {duration_sec:.2f}s")
print(f"Average WPM: {wpm:.1f}")
print(f"File Size: {file_size_mb:.2f} MB")
display(Audio(out_path))

In [ ]:
chapter_text = """## Introduction
Welcome to the chapter splitting feature. This is a short intro.

## Chapter 1
In the beginning, we created this tool to help narrate texts. It splits by markdown headers.

## Conclusion
And that wraps up the chapter splitting test!"""

chapters = re.split(r'(?=^##\s)', chapter_text, flags=re.MULTILINE)
for i, chap in enumerate(chapters):
    chap = chap.strip()
    if not chap: continue
    print(f"--- Generating Chapter {i} ---")
    chap_audio, sr = model.generate_voice_design(chap, "en", INSTRUCT)
    chap_path = f"{OUTPUT_DIR}/chapter_{i}.wav"
    sf.write(chap_path, chap_audio, sr)
    display(Audio(chap_path))

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("document_audio", "zip", OUTPUT_DIR)
print("Downloading document_audio.zip...")
files.download("document_audio.zip")